# Google Trend Archive – Data Loader Demo

**Dataset:** [aurman/GoogleTrendArchive](https://huggingface.co/datasets/aurman/GoogleTrendArchive)  
**DOI:** [10.57967/hf/7531](https://doi.org/10.57967/hf/7531)  
**License:** CC-BY-4.0

This notebook walks you through:
1. Loading the dataset from Hugging Face
2. Parsing timestamps and computing trend duration
3. Filtering by country / region
4. Exploring top queries and search volume buckets
5. Plotting trends over time and duration distributions

In [ ]:
# Install dependencies (run once)
# !pip install datasets pandas matplotlib

## 1. Load the Dataset

In [ ]:
from datasets import load_dataset
import pandas as pd

# Streaming mode — no full download required, great for exploration
ds = load_dataset(
    "aurman/GoogleTrendArchive",
    split="train",
    streaming=True,
)

# Preview the first 5 rows
for i, row in enumerate(ds.take(5)):
    print(row)

In [ ]:
# Build a working sample DataFrame from the first 10,000 rows
# (Replace with pd.read_csv(...) if you have the full CSV downloaded locally)

sample = list(ds.take(10_000))
df = pd.DataFrame(sample)

print(f"Shape: {df.shape}")
df.head()

## 2. Parse Timestamps & Compute Trend Duration

In [ ]:
df["Started"] = pd.to_datetime(df["Started"], utc=True, errors="coerce")
df["Ended"]   = pd.to_datetime(df["Ended"],   utc=True, errors="coerce")
df["duration_minutes"] = (df["Ended"] - df["Started"]).dt.total_seconds() / 60

df[["Trends", "location", "Started", "Ended", "duration_minutes"]].head(10)

In [ ]:
df["duration_minutes"].describe().round(1)

## 3. Filter by Location

The `location` column uses ISO country / region codes.  
Country examples: `US`, `DE`, `FR`, `JP`, `BR`  
Sub-national examples: `US-CA` (California), `AU-VIC` (Victoria), `DE-BY` (Bavaria)

In [ ]:
TARGET = "US"
df_loc = df[df["location"] == TARGET]
print(f"Rows for '{TARGET}': {len(df_loc):,}")
df_loc.head()

## 4. Top Trending Queries

In [ ]:
df_loc["Trends"].value_counts().head(15)

## 5. Search Volume Bucket Distribution

In [ ]:
df["Search volume"].value_counts()

## 6. Trends Over Time

In [ ]:
import matplotlib.pyplot as plt

daily_counts = (
    df.dropna(subset=["Started"])
    .set_index("Started")
    .resample("D")
    .size()
    .rename("num_trends")
)

fig, ax = plt.subplots(figsize=(13, 4))
daily_counts.plot(ax=ax, linewidth=1.5, color="#e63946")
ax.set_title("Daily Trending Search Instances (sample)", fontsize=13)
ax.set_xlabel("Date")
ax.set_ylabel("Number of trends")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Trend Duration Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
df["duration_minutes"].clip(upper=1440).hist(bins=50, ax=ax, color="#457b9d", edgecolor="white")
ax.set_title("Trend Duration Distribution (capped at 24 h)", fontsize=13)
ax.set_xlabel("Duration (minutes)")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

## 8. Loading the Full Preprocessed CSV (locally)

If you downloaded `googletrendarchive_preprocessed.csv` from Hugging Face:

In [ ]:
# df_full = pd.read_csv("googletrendarchive_preprocessed.csv", low_memory=False)
# df_full["Started"] = pd.to_datetime(df_full["Started"], utc=True)
# df_full["Ended"]   = pd.to_datetime(df_full["Ended"],   utc=True)
# print(df_full.shape)
# df_full.head()

---

## Citation

```bibtex
@dataset{urman2026googletrendarchive,
  title     = {Google Trend Archive: Global Real-Time Search Trends},
  author    = {Urman, Aleksandra and Hann{\'{a}}k, Anik{\'{o}} and Baumann, Joachim},
  year      = {2026},
  publisher = {Hugging Face},
  doi       = {10.57967/hf/7531},
  url       = {https://huggingface.co/datasets/aurman/GoogleTrendArchive}
}
```